In [43]:
import sys
sys.path.append("..")  # so we can import from src/

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn as sk
from sklearn.model_selection import StratifiedGroupKFold

from src.data_loader import load_raw_data, load_id_mappings, add_target

In [44]:
df = load_raw_data()
df.set_index('encounter_id', inplace=True)
df.reset_index(drop=True)
df.shape

(101766, 49)

In [45]:
#removing expired and hospice patients

exp_hsp_pat = [11, 13, 14, 19, 20]
df = df[~df['discharge_disposition_id'].isin(exp_hsp_pat)]
df.shape

(99343, 49)

In [46]:
#dropping weight as it has 98569 rows as blank
df.drop('weight', axis=1, inplace=True)
df.shape

(99343, 48)

In [47]:
#filling missing values with unknown
cols = ["race", "payer_code", "medical_specialty", "diag_1", "diag_2", "diag_3"]
df[cols] = df[cols].fillna("Unknown")

In [48]:
#examide and citoglipton are "No" for all 99,343 patients, so we are dropping them as it constant for almost all patients
unq_values = df.nunique()
const_columns = unq_values[unq_values == 1].index.tolist()
print(const_columns)
df = df.drop(columns=const_columns)

['examide', 'citoglipton']


In [49]:
#converting age map into numeric
age_map = {"[0-10)":1,"[10-20)":2,"[20-30)":3,"[30-40)":4,"[40-50)":5,"[50-60)":6,"[60-70)":7,"[70-80)":8,"[80-90)":9,"[90-100)":10}
df['age'] = df['age'].map(age_map)
#df.head()
#df['age'].isna().sum()


In [50]:
#creating the target column
Y = (df['readmitted'] == '<30').astype(int)
print(Y)
df.drop('readmitted', axis=1, inplace=True)
X = df
X.shape

encounter_id
2278392      0
149190       0
64410        0
500364       0
16680        0
            ..
443847548    0
443847782    0
443854148    0
443857166    0
443867222    0
Name: readmitted, Length: 99343, dtype: int64


(99343, 45)

In [60]:
#dropping unknown
#print(X["gender"].value_counts())
X = X[X["gender"] != "Unknown/Invalid"]
print(X["gender"].value_counts())
Y = Y.loc[X.index]

#print(X['gender'].nunique())

gender
0    53454
1    45886
Name: count, dtype: int64


In [61]:
#converting categorical columns to binary columns
X["gender"] = X["gender"].map({"Female": 0, "Male": 1})
X["change"] = X["change"].map({"No": 0, "Ch": 1})
X["diabetesMed"] = X["diabetesMed"].map({"No": 0, "Yes": 1})
print(X[["gender", "change", "diabetesMed"]].isna().sum())   # all 0

gender         99340
change         99340
diabetesMed    99340
dtype: int64


In [68]:
from sklearn.model_selection import GroupShuffleSplit

groups = X["patient_nbr"]
gss = GroupShuffleSplit(n_splits=1,test_size=0.3,random_state=42)
train_idx,test_idx = next(gss.split(X, Y,groups))

In [69]:
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
Y_train, Y_test = Y.iloc[train_idx], Y.iloc[test_idx]
group_train = groups.iloc[train_idx]

In [72]:
print(X_train.shape, X_test.shape)
print(len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))
print(Y_train.mean(),Y_test.mean())

(69613, 45) (29727, 45)
0
0.1137000272937526 0.11434049853668382


In [73]:
#Drop the patient id from the features
X_train = X_train.drop(columns="patient_nbr")
X_test = X_test.drop(columns="patient_nbr")

In [75]:
#Save to Parquet
from src.data_loader import PROCESSED_DIR

X_train.to_parquet(PROCESSED_DIR / "X_train.parquet")
X_test.to_parquet(PROCESSED_DIR / "X_test.parquet")
Y_train.to_frame("readmitted_30").to_parquet(PROCESSED_DIR / "y_train.parquet")
Y_test.to_frame("readmitted_30").to_parquet(PROCESSED_DIR / "y_test.parquet")
group_train.to_frame("patient_nbr").to_parquet(PROCESSED_DIR / "groups_train.parquet")